# ROGII Wellbore Geology - LSTM Training

Train a Bidirectional LSTM to predict **TVT** from horizontal well logs.

**Features (6):** `MD, X, Y, Z, GR, TVT_input`

**Label:** `TVT`

**Runtime**: Kaggle GPU (T4/P100) - Keras 3 + JAX backend

**Author**: Samir Attrah

In [1]:
# Cell 1: Environment & Imports
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
from keras import layers, callbacks, regularizers
import jax.numpy as jnp
import numpy as np
import polars as pl
import glob, pickle, warnings, random
warnings.filterwarnings("ignore")

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Keras backend : {keras.backend.backend()}")
print(f"Working dir   : {os.getcwd()}")


2026-05-23 02:32:42.069537: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779492762.114674  147889 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779492762.130253  147889 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Keras version : 3.12.0
Keras backend : jax
Working dir   : /home/samer/Documents/competitions/ROGII/notebooks


In [2]:
# Cell 2: Auto-detect dataset path

def find_data_dir():
    """Searches common Kaggle mount points for the ROGII dataset.

    Checks competition path, regular dataset path, and local fallback.

    Returns:
        Absolute path to the dataset root directory.

    Raises:
        FileNotFoundError: If no valid dataset is found.
    """
    # Ordered list of candidate paths to check
    candidates = [
        "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
        "/kaggle/input/rogii-wellbore-geology-prediction",
        "/home/samer/Documents/competitions/ROGII/dataset",
    ]

    # Also scan /kaggle/input/ for any match
    for scan_root in ["/kaggle/input", "/kaggle/input/competitions"]:
        if os.path.isdir(scan_root):
            for entry in os.listdir(scan_root):
                full = os.path.join(scan_root, entry)
                if os.path.isdir(full) and full not in candidates:
                    candidates.append(full)

    print("Searching for ROGII dataset...")
    for path in candidates:
        if not os.path.isdir(path):
            print(f"  X {path}  (does not exist)")
            continue

        contents = os.listdir(path)
        has_train = "train" in contents and os.path.isdir(os.path.join(path, "train"))
        has_test = "test" in contents and os.path.isdir(os.path.join(path, "test"))

        # Count CSVs in train/ and test/
        n_train = 0
        if has_train:
            n_train = len(glob.glob(os.path.join(path, "train", "*__horizontal_well.csv")))
        n_test = 0
        if has_test:
            n_test = len(glob.glob(os.path.join(path, "test", "*__horizontal_well.csv")))

        print(f"  -> {path}")
        print(f"    train/: {n_train} wells | test/: {n_test} wells")

        if n_train > 0 or n_test > 0:
            print(f"  V Using this path as DATA_DIR")
            return path
        else:
            print(f"    (no well CSVs found, skipping)")

    raise FileNotFoundError(
        "Could not find ROGII dataset. Checked:\n"
        + "\n".join(f"  - {c}" for c in candidates)
    )

DATA_DIR = find_data_dir()


Searching for ROGII dataset...
  X /kaggle/input/competitions/rogii-wellbore-geology-prediction  (does not exist)
  X /kaggle/input/rogii-wellbore-geology-prediction  (does not exist)
  -> /home/samer/Documents/competitions/ROGII/dataset
    train/: 773 wells | test/: 3 wells
  V Using this path as DATA_DIR


In [3]:
# Cell 3: Configuration

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle") else "/tmp/rogii_output"
os.makedirs(OUT_DIR, exist_ok=True)

CONFIG = {
    "seed": 42,
    "data_dir": DATA_DIR,
    "model_path": f"{OUT_DIR}/lstm_tvt_model.keras",
    "scaler_path": f"{OUT_DIR}/scaler_params.pkl",
    "window_size": 6,
    "stride": 4,
    "lstm_units_1": 64,
    "lstm_units_2": 32,
    "lstm_units_3": 32,
    "lstm_units_4": 32,
    "lstm_units_5": 32,
    "lstm_units_6": 32,
    "kr_rate": 1e-5,
    "dense_units": 16,
    "dropout": 0.30,
    "epochs": 100,
    "batch_size": 16,
    "lr": 1e-3,
    "val_ratio": 0.20,
    "max_wells": None,
    "gcn": 1,
}

# --- NEW feature set: 6 raw columns from horizontal_well.csv ---
FEATURE_COLS = [
    "MD", "X", "Y", "Z", "GR", "TVT_input",
]
TARGET = "TVT"

print(f"Data dir   : {CONFIG['data_dir']}")
print(f"Output dir : {OUT_DIR}")
print(f"Model path : {CONFIG['model_path']}")
print(f"Features   : {len(FEATURE_COLS)} columns -> {FEATURE_COLS}")
print(f"Label      : {TARGET}")


Data dir   : /home/samer/Documents/competitions/ROGII/dataset
Output dir : /tmp/rogii_output
Model path : /tmp/rogii_output/lstm_tvt_model.keras
Features   : 6 columns -> ['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input']
Label      : TVT


In [4]:
# Cell 4: Data helpers

def well_ids(data_dir, split="train"):
    """Returns sorted well IDs for a split.

    Args:
        data_dir: Root dataset directory.
        split: Either 'train' or 'test'.

    Returns:
        Sorted list of well ID strings.
    """
    pattern = os.path.join(data_dir, split, "*__horizontal_well.csv")
    ids = sorted(os.path.basename(f).split("__")[0] for f in glob.glob(pattern))
    print(f"  Found {len(ids)} wells in {split}/")
    return ids


def load_well(data_dir, wid, split="train"):
    """Loads one horizontal well CSV as Polars DataFrame.

    Args:
        data_dir: Root dataset directory.
        wid: Well identifier string.
        split: Either 'train' or 'test'.

    Returns:
        Polars DataFrame with all columns from the CSV.
    """
    path = os.path.join(data_dir, split, f"{wid}__horizontal_well.csv")
    return pl.read_csv(path, infer_schema_length=10000)


def preprocess(df):
    """Preprocess a horizontal well DataFrame for the 6-feature set.

    Handles null values in GR and TVT_input (interpolation + forward/backward fill).
    All remaining nulls are filled with 0.0.

    Args:
        df: Raw Polars DataFrame from load_well().

    Returns:
        Polars DataFrame with clean feature columns.
    """
    # GR and TVT_input: linear interpolation then forward/backward fill
    for col in ["GR", "TVT_input"]:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col).interpolate()
                  .fill_null(strategy="forward").fill_null(strategy="backward")
                  .fill_null(0.0)
            )

    return df


def make_seqs(feats, tgts, ws, stride):
    """Creates sliding-window sequences from feature and target arrays.

    Args:
        feats: 2D numpy array of shape (n_rows, n_features).
        tgts: 1D numpy array of shape (n_rows,).
        ws: Window size (sequence length).
        stride: Step between consecutive windows.

    Returns:
        Tuple of (X, y) where X has shape (n_seqs, ws, n_features)
        and y has shape (n_seqs,). Each y value corresponds to the
        target at the last timestep of its window.
    """
    idx = range(0, len(feats) - ws, stride)
    X = np.empty((len(idx), ws, feats.shape[1]), dtype=np.float32)
    y = np.empty(len(idx), dtype=np.float32)
    for i, s in enumerate(idx):
        X[i] = feats[s:s+ws]
        y[i] = tgts[s+ws-1]
    return X, y


# --- Verify data access ---
print("--- Verifying data access ---")
ids = well_ids(CONFIG["data_dir"], "train")
test_ids_check = well_ids(CONFIG["data_dir"], "test")

sample = load_well(CONFIG["data_dir"], ids[0])
print(f"\nSample well: {ids[0]}")
print(f"  Columns: {sample.columns}")
print(f"  Shape  : {sample.shape}")
print(f"  GR nulls : {sample['GR'].null_count()}")
print(f"  TVT nulls: {sample['TVT'].null_count()}")

sample_pp = preprocess(sample)
missing_feats = [c for c in FEATURE_COLS if c not in sample_pp.columns]
if missing_feats:
    print(f"  Missing features after preprocessing: {missing_feats}")
else:
    print(f"  All {len(FEATURE_COLS)} features available")
    print(f"  GR nulls after preprocess: {sample_pp['GR'].null_count()}")


--- Verifying data access ---
  Found 773 wells in train/
  Found 3 wells in test/

Sample well: 000d7d20
  Columns: ['MD', 'X', 'Y', 'Z', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA', 'TVT', 'GR', 'TVT_input']
  Shape  : (5278, 13)
  GR nulls : 2258
  TVT nulls: 0
  All 6 features available
  GR nulls after preprocess: 0


In [5]:
# Cell 5: Prepare training data

def prepare_data(cfg):
    """Load all training wells, preprocess, create sequences, normalise.

    Splits wells into train/val sets, builds sliding-window sequences,
    and computes Z-score normalisation parameters for both features
    and the TVT target.

    Args:
        cfg: Configuration dictionary with keys 'data_dir',
             'max_wells', 'val_ratio', 'window_size', 'stride'.

    Returns:
        Tuple of (X_train, y_train, X_val, y_val, y_val_raw, scaler)
        where scaler is a dict with normalisation parameters.
    """
    ids_all = well_ids(cfg["data_dir"], "train")
    if cfg["max_wells"]:
        ids_all = ids_all[:cfg["max_wells"]]
        print(f"  (limited to {cfg['max_wells']} wells)")
    ws = cfg["window_size"]

    # Well-level train/val split
    np.random.seed(cfg.get("seed", 42))
    n_val = max(1, int(len(ids_all) * cfg["val_ratio"]))
    val_set = set(np.random.permutation(len(ids_all))[:n_val])
    print(f"  Train wells: {len(ids_all) - n_val}, Val wells: {n_val}")

    Xt, yt, Xv, yv = [], [], [], []
    skipped = 0
    for i, wid in enumerate(ids_all):
        if (i+1) % 100 == 0:
            print(f"  Processing well {i+1}/{len(ids_all)}...")
        try:
            df = preprocess(load_well(cfg["data_dir"], wid))

            # Filter rows where TVT is available
            df = df.filter(pl.col(TARGET).is_not_null())
            if len(df) <= ws:
                skipped += 1
                continue

            # Extract features and target
            f = np.nan_to_num(
                df.select(FEATURE_COLS).to_numpy().astype(np.float32)
            )
            t = np.nan_to_num(
                df.select(TARGET).to_numpy().ravel().astype(np.float32)
            )

            X, y = make_seqs(f, t, ws, cfg["stride"])
            (Xv if i in val_set else Xt).append(X)
            (yv if i in val_set else yt).append(y)
        except Exception as e:
            skipped += 1
            print(f"  Skip {wid}: {e}")

    Xt, yt = np.concatenate(Xt), np.concatenate(yt)
    Xv, yv = np.concatenate(Xv), np.concatenate(yv)

    print(f"\n  Wells skipped: {skipped}")
    print(f"  Train sequences: {Xt.shape}")
    print(f"  Val sequences  : {Xv.shape}")

    # Z-score normalisation (per feature)
    mu = Xt.mean(axis=(0, 1))
    sigma = Xt.std(axis=(0, 1)) + 1e-8
    Xt = (Xt - mu) / sigma
    Xv = (Xv - mu) / sigma

    # Target normalisation
    ym, ys = yt.mean(), yt.std() + 1e-8
    print(f"  Target mean: {ym:.2f}, std: {ys:.2f}")

    scaler = {
        "feat_mean": mu,
        "feat_std": sigma,
        "target_mean": ym,
        "target_std": ys,
        "feature_cols": FEATURE_COLS,
    }
    return Xt, (yt - ym) / ys, Xv, (yv - ym) / ys, yv, scaler


print("--- Loading and preparing training data ---")
X_train, y_train, X_val, y_val, y_val_raw, scaler = prepare_data(CONFIG)

with open(CONFIG["scaler_path"], "wb") as f:
    pickle.dump(scaler, f)
print(f"\nScaler saved: {CONFIG['scaler_path']}")


--- Loading and preparing training data ---
  Found 773 wells in train/
  Train wells: 619, Val wells: 154
  Processing well 100/773...
  Processing well 200/773...
  Processing well 300/773...
  Processing well 400/773...
  Processing well 500/773...
  Processing well 600/773...
  Processing well 700/773...

  Wells skipped: 0
  Train sequences: (1013030, 6, 6)
  Val sequences  : (259168, 6, 6)
  Target mean: 11493.00, std: 636.86

Scaler saved: /tmp/rogii_output/scaler_params.pkl


In [8]:
# Cell 6: Build & train BiLSTM model

def build_model(input_shape, cfg):
    """Builds a Bidirectional LSTM regression model for TVT prediction.

    Architecture: BiLSTM -> BN -> BiLSTM -> BN -> Dense -> Dropout -> Dense(1)

    Args:
        input_shape: Tuple of (window_size, n_features).
        cfg: Configuration dict with model hyperparameters.

    Returns:
        Compiled Keras model.
    """
    inp = keras.Input(shape=input_shape)
    x = layers.Bidirectional(
        layers.LSTM(cfg["lstm_units_1"], return_sequences=True, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),
                    dropout=cfg["dropout"]), name="bilstm1")(inp)
    # x = layers.BatchNormalization()(x)
    x = layers.Bidirectional(
        layers.LSTM(cfg["lstm_units_2"], return_sequences=True, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),
                    dropout=cfg["dropout"]), name="bilstm2")(x)
    # x = layers.BatchNormalization()(x)
    x = layers.Bidirectional(
        layers.LSTM(cfg["lstm_units_3"], return_sequences=True, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),
                    dropout=cfg["dropout"]), name="bilstm3")(x)
    # x = layers.BatchNormalization()(x)
    x = layers.Bidirectional(
        layers.LSTM(cfg["lstm_units_4"], return_sequences=True, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),
                    dropout=cfg["dropout"]), name="bilstm4")(x)
    # x = layers.BatchNormalization()(x)
    x = layers.Bidirectional(
        layers.LSTM(cfg["lstm_units_5"], return_sequences=True, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),
                    dropout=cfg["dropout"]), name="bilstm5")(x)
    # x = layers.BatchNormalization()(x)
    x = layers.Bidirectional(
        layers.LSTM(cfg["lstm_units_6"], return_sequences=False, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),
                    dropout=cfg["dropout"]), name="bilstm6")(x)
    # x = layers.BatchNormalization()(x)
    x = layers.Dense(cfg["dense_units"], activation="relu")(x)
    x = layers.Dropout(cfg["dropout"])(x)
    out = layers.Dense(1)(x)
    m = keras.Model(inp, out, name="BiLSTM_TVT")
    m.compile(
        optimizer=keras.optimizers.Adam(cfg["lr"], global_clipnorm=cfg["gcn"]),
        loss="mse",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    return m


input_shape = (CONFIG["window_size"], len(FEATURE_COLS))
print(f"Building model with input shape: {input_shape}")
model = build_model(input_shape, CONFIG)
model.summary()

cbs = [
    # callbacks.EarlyStopping(
    #     monitor="val_rmse", patience=20,
    #     restore_best_weights=True, mode="min"),
    callbacks.ModelCheckpoint(
        CONFIG["model_path"], monitor="val_rmse",
        save_best_only=True, mode="min"),
    callbacks.ReduceLROnPlateau(
        monitor="val_rmse", factor=0.5,
        patience=3, min_lr=1e-6),
]

print(f"\nStarting training: {CONFIG['epochs']} epochs, batch={CONFIG['batch_size']}")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    callbacks=cbs,
)
print(f"\nTraining complete. Best model saved to {CONFIG['model_path']}")


Building model with input shape: (6, 6)


Model: "BiLSTM_TVT"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 6, 6)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm1 (Bidirectional)         │ (None, 6, 128)         │        36,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm2 (Bidirectional)         │ (None, 6, 64)          │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm3 (Bidirectional)         │ (None, 6, 64)          │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm4 (Bidirectional)         │ (None, 6, 64)          │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm5 (Bidirectional)         │ (None, 6, 64)          │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm6 (Bidirectional)         │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 177,953 (695.13 KB)

 Trainable params: 177,953 (695.13 KB)

 Non-trainable params: 0 (0.00 B)


Starting training: 100 epochs, batch=16
Epoch 1/100
 2330/63315 ━━━━━━━━━━━━━━━━━━━━ 8:58 9ms/step - loss: 0.2014 - rmse: 0.4306

KeyboardInterrupt: 

In [ ]:
# Cell 7: Evaluate & plot

import matplotlib.pyplot as plt

# Loss & RMSE curves
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(history.history["loss"], label="train")
ax[0].plot(history.history["val_loss"], label="val")
ax[0].set_title("MSE Loss"); ax[0].legend()
ax[1].plot(history.history["rmse"], label="train")
ax[1].plot(history.history["val_rmse"], label="val")
ax[1].set_title("RMSE"); ax[1].legend()
plt.tight_layout(); plt.show()

# Invert normalisation and compute original-scale metrics
yp = (
    model.predict(X_val, batch_size=512).ravel()
    * scaler["target_std"]
    + scaler["target_mean"]
)
rmse = float(jnp.sqrt(jnp.mean((yp - y_val_raw)**2)))
mae = float(jnp.mean(jnp.abs(yp - y_val_raw)))
print(f"\nVal RMSE (original scale): {rmse:.4f}")
print(f"Val MAE  (original scale): {mae:.4f}")

# Scatter: predicted vs actual TVT
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_val_raw[:5000], yp[:5000], alpha=0.15, s=4)
lim = [min(y_val_raw.min(), yp.min()), max(y_val_raw.max(), yp.max())]
ax.plot(lim, lim, "r--")
ax.set_xlabel("True TVT")
ax.set_ylabel("Pred TVT")
ax.set_title(f"Val RMSE={rmse:.2f}")
plt.tight_layout(); plt.show()

print(f"Model saved: {CONFIG['model_path']}")
print(f"Scaler saved: {CONFIG['scaler_path']}")
print(f"Feature cols stored in scaler: {scaler['feature_cols']}")
print("Proceed to 02_lstm_inference.ipynb")
